# Notebook 13 — Final Untouched Test, Statistical Significance & Research Conclusions

This notebook performs the **final untouched chronological evaluation** after Notebook 12 froze the research configuration.

### Core rule

The final test period is not used for:
- model selection
- feature selection
- threshold tuning
- hyperparameter tuning
- cost tuning

The frozen candidate model and frozen trading configuration from Notebook 12 are evaluated exactly as documented.

### Outputs

- Final untouched classification metrics
- Final untouched trading metrics
- Buy-and-hold benchmark
- Hit-rate significance test
- Return significance test
- Bootstrap Sharpe confidence interval
- Bootstrap strategy-vs-benchmark return difference
- Final research conclusion
- Final evaluation report


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    XGBClassifier = None

try:
    from scipy.stats import binomtest, ttest_rel
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "data").exists():
    for c in [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]:
        if (c / "data").exists() and (c / "notebooks").exists():
            ROOT = c
            break

MASTER_PATH = ROOT / "data" / "raw" / "sp500_1950_present.csv"
WF_PATH = ROOT / "data" / "interim" / "sp500_walk_forward_predictions.parquet"
CONFIG_PATH = ROOT / "models" / "frozen" / "sp500_frozen_research_config.json"

INTERIM = ROOT / "data" / "interim"
FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
REPORTS = ROOT / "reports" / "generated"
MODEL_DIR = ROOT / "models" / "frozen"

for x in [INTERIM, FIGURES, TABLES, REPORTS, MODEL_DIR]:
    x.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252
FINAL_TEST_FRACTION = 0.15
RANDOM_SEED = 42
BOOTSTRAP_SAMPLES = 3000

print("Project root:", ROOT)
print("SciPy available:", SCIPY_AVAILABLE)
print("XGBoost available:", XGB_AVAILABLE)


In [ ]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"{CONFIG_PATH} not found. Run Notebook 12 first."
    )

frozen_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

candidate_model = frozen_config["candidate_model"]
frozen_threshold = float(
    frozen_config["probability_threshold"]
)
frozen_cost_bps = float(
    frozen_config["transaction_cost_bps"]
)
frozen_slippage_bps = float(
    frozen_config["slippage_bps"]
)

print(json.dumps(frozen_config, indent=2))


In [ ]:
def build_features(raw):
    df = raw.copy()

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    for c in [
        "Open","High","Low","Close",
        "Adj.Close","Volume"
    ]:
        df[c] = pd.to_numeric(
            df[c],
            errors="coerce"
        )

    df = (
        df.sort_values("Date")
        .reset_index(drop=True)
    )

    df["return_1d"] = df["Close"].pct_change()
    df["return_5d"] = df["Close"].pct_change(5)
    df["return_21d"] = df["Close"].pct_change(21)
    df["return_63d"] = df["Close"].pct_change(63)
    df["return_126d"] = df["Close"].pct_change(126)
    df["return_252d"] = df["Close"].pct_change(252)

    df["volatility_5d"] = (
        df["return_1d"].rolling(5).std() *
        np.sqrt(252)
    )
    df["volatility_21d"] = (
        df["return_1d"].rolling(21).std() *
        np.sqrt(252)
    )
    df["volatility_63d"] = (
        df["return_1d"].rolling(63).std() *
        np.sqrt(252)
    )

    df["sma_20"] = df["Close"].rolling(20).mean()
    df["sma_50"] = df["Close"].rolling(50).mean()
    df["sma_200"] = df["Close"].rolling(200).mean()

    df["price_to_sma_20"] = (
        df["Close"] / df["sma_20"] - 1
    )
    df["price_to_sma_50"] = (
        df["Close"] / df["sma_50"] - 1
    )
    df["price_to_sma_200"] = (
        df["Close"] / df["sma_200"] - 1
    )
    df["sma_50_vs_sma_200"] = (
        df["sma_50"] / df["sma_200"] - 1
    )

    df["range_pct"] = (
        (df["High"] - df["Low"]) /
        df["Close"]
    )
    df["intraday_return"] = (
        df["Close"] / df["Open"] - 1
    )
    df["overnight_return"] = (
        df["Open"] / df["Close"].shift(1) - 1
    )

    df["volume_ratio_20"] = (
        df["Volume"] /
        df["Volume"].rolling(20).mean()
    )
    df["volume_ratio_63"] = (
        df["Volume"] /
        df["Volume"].rolling(63).mean()
    )

    df["return_1d_lag1"] = df["return_1d"].shift(1)
    df["return_1d_lag2"] = df["return_1d"].shift(2)
    df["return_1d_lag3"] = df["return_1d"].shift(3)
    df["return_1d_lag5"] = df["return_1d"].shift(5)
    df["return_1d_lag10"] = df["return_1d"].shift(10)

    df["volatility_21d_lag1"] = (
        df["volatility_21d"].shift(1)
    )
    df["range_pct_lag1"] = (
        df["range_pct"].shift(1)
    )
    df["volume_ratio_20_lag1"] = (
        df["volume_ratio_20"].shift(1)
    )

    df["next_day_return"] = (
        df["Close"].shift(-1) /
        df["Close"] - 1
    )

    df["target"] = (
        df["next_day_return"] > 0
    ).astype(int)

    feature_columns = [
        "return_1d",
        "return_5d",
        "return_21d",
        "return_63d",
        "return_126d",
        "return_252d",
        "volatility_5d",
        "volatility_21d",
        "volatility_63d",
        "price_to_sma_20",
        "price_to_sma_50",
        "price_to_sma_200",
        "sma_50_vs_sma_200",
        "range_pct",
        "intraday_return",
        "overnight_return",
        "volume_ratio_20",
        "volume_ratio_63",
        "return_1d_lag1",
        "return_1d_lag2",
        "return_1d_lag3",
        "return_1d_lag5",
        "return_1d_lag10",
        "volatility_21d_lag1",
        "range_pct_lag1",
        "volume_ratio_20_lag1",
    ]

    modeling = df[
        ["Date", "Close", "next_day_return", "target"] +
        feature_columns
    ].dropna(
        subset=feature_columns +
        ["next_day_return", "target"]
    ).reset_index(drop=True)

    return df, modeling, feature_columns


raw = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

full_df, modeling_df, feature_columns = build_features(raw)

print("Raw rows:", len(full_df))
print("Modeling rows:", len(modeling_df))
print("Features:", len(feature_columns))


In [ ]:
# Reserve the final 15% chronologically.
# The final block is never used for model selection or tuning.

split_index = int(
    len(modeling_df) *
    (1 - FINAL_TEST_FRACTION)
)

train_df = modeling_df.iloc[
    :split_index
].copy()

test_df = modeling_df.iloc[
    split_index:
].copy()

assert train_df["Date"].max() < test_df["Date"].min()

print("Training rows:", len(train_df))
print("Final untouched test rows:", len(test_df))
print(
    "Training range:",
    train_df["Date"].min().date(),
    "→",
    train_df["Date"].max().date()
)
print(
    "Final test range:",
    test_df["Date"].min().date(),
    "→",
    test_df["Date"].max().date()
)


In [ ]:
def create_model(name):
    if name == "Logistic Regression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                random_state=42
            ))
        ])

    if name == "Random Forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=250,
                max_depth=8,
                min_samples_leaf=10,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            ))
        ])

    if name == "HistGradientBoosting":
        return Pipeline([
            ("model", HistGradientBoostingClassifier(
                max_iter=250,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42
            ))
        ])

    if name == "XGBoost":
        if not XGB_AVAILABLE:
            raise ImportError(
                "XGBoost is required for the frozen candidate model."
            )
        return XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=5,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

    raise ValueError(
        f"Unknown candidate model: {name}"
    )


model = create_model(candidate_model)

model.fit(
    train_df[feature_columns],
    train_df["target"]
)

test_probability = model.predict_proba(
    test_df[feature_columns]
)[:, 1]

test_prediction = (
    test_probability >= frozen_threshold
).astype(int)

print("Frozen model trained only on pre-test observations.")


In [ ]:
classification_metrics = {
    "accuracy": accuracy_score(
        test_df["target"],
        test_prediction
    ),
    "precision": precision_score(
        test_df["target"],
        test_prediction,
        zero_division=0
    ),
    "recall": recall_score(
        test_df["target"],
        test_prediction,
        zero_division=0
    ),
    "f1": f1_score(
        test_df["target"],
        test_prediction,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        test_df["target"],
        test_probability
    )
}

final_predictions = test_df[
    [
        "Date",
        "Close",
        "next_day_return",
        "target"
    ]
].copy()

final_predictions["probability_up"] = test_probability
final_predictions["prediction"] = test_prediction

display(
    pd.DataFrame([classification_metrics])
)


In [ ]:
def backtest_final(frame):
    frame = frame.copy()

    position = (
        frame["probability_up"] >=
        frozen_threshold
    ).astype(float)

    gross = (
        position *
        frame["next_day_return"]
    )

    turnover = (
        position
        .diff()
        .abs()
        .fillna(position.abs())
    )

    total_cost_rate = (
        frozen_cost_bps +
        frozen_slippage_bps
    ) / 10000

    costs = (
        turnover *
        total_cost_rate
    )

    net = gross - costs

    equity = (
        1 + net
    ).cumprod()

    drawdown = (
        equity /
        equity.cummax() -
        1
    )

    years = len(net) / TRADING_DAYS
    cagr = (
        equity.iloc[-1] ** (1 / years) - 1
        if years > 0 and equity.iloc[-1] > 0
        else np.nan
    )

    vol = (
        net.std(ddof=1) *
        np.sqrt(TRADING_DAYS)
    )

    sharpe = (
        net.mean() /
        net.std(ddof=1) *
        np.sqrt(TRADING_DAYS)
        if net.std(ddof=1) > 0
        else np.nan
    )

    downside = net[net < 0]

    if len(downside):
        downside_dev = np.sqrt(
            np.mean(downside ** 2)
        )
        sortino = (
            net.mean() /
            downside_dev *
            np.sqrt(TRADING_DAYS)
            if downside_dev > 0
            else np.nan
        )
    else:
        sortino = np.inf

    result = frame[
        [
            "Date",
            "Close",
            "next_day_return",
            "target",
            "probability_up",
            "prediction"
        ]
    ].copy()

    result["position"] = position
    result["turnover"] = turnover
    result["cost"] = costs
    result["gross_return"] = gross
    result["net_return"] = net
    result["equity"] = equity
    result["drawdown"] = drawdown

    summary = {
        "cumulative_return": equity.iloc[-1] - 1,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": drawdown.min(),
        "Calmar": (
            cagr / abs(drawdown.min())
            if drawdown.min() < 0
            else np.nan
        ),
        "win_rate": (
            (net[position != 0] > 0).mean()
            if (position != 0).any()
            else np.nan
        ),
        "trades": int(
            (turnover > 0).sum()
        ),
        "turnover": turnover.sum(),
        "average_position": position.mean()
    }

    return result, summary


final_frame, final_strategy_metrics = backtest_final(
    final_predictions
)

print("Final untouched strategy metrics:")
display(
    pd.DataFrame([final_strategy_metrics])
)


In [ ]:
benchmark_returns = (
    final_frame["next_day_return"]
    .fillna(0)
)

benchmark_equity = (
    1 + benchmark_returns
).cumprod()

benchmark_dd = (
    benchmark_equity /
    benchmark_equity.cummax() -
    1
)

years = len(benchmark_returns) / TRADING_DAYS

benchmark_cagr = (
    benchmark_equity.iloc[-1] ** (1 / years) - 1
    if years > 0 and benchmark_equity.iloc[-1] > 0
    else np.nan
)

benchmark_sharpe = (
    benchmark_returns.mean() /
    benchmark_returns.std(ddof=1) *
    np.sqrt(TRADING_DAYS)
    if benchmark_returns.std(ddof=1) > 0
    else np.nan
)

benchmark_summary = {
    "cumulative_return": benchmark_equity.iloc[-1] - 1,
    "CAGR": benchmark_cagr,
    "annualized_volatility": (
        benchmark_returns.std(ddof=1) *
        np.sqrt(TRADING_DAYS)
    ),
    "Sharpe": benchmark_sharpe,
    "max_drawdown": benchmark_dd.min()
}

display(
    pd.DataFrame([
        {"strategy": "Frozen Strategy", **final_strategy_metrics},
        {"strategy": "Buy & Hold", **benchmark_summary}
    ])
)


In [ ]:
# Hit-rate significance against a 50% random-direction null.

correct = (
    final_predictions["prediction"] ==
    final_predictions["target"]
)

n = int(correct.sum() + (~correct).sum())
successes = int(correct.sum())
hit_rate = successes / n

if SCIPY_AVAILABLE:
    hit_test = binomtest(
        successes,
        n,
        p=0.50,
        alternative="greater"
    )
    hit_pvalue = hit_test.pvalue
else:
    # Normal approximation with continuity correction.
    z = (
        (successes - n * 0.5 - 0.5) /
        np.sqrt(n * 0.25)
    )
    from math import erf, sqrt
    hit_pvalue = 0.5 * (1 - erf(z / sqrt(2)))

hit_significance = pd.DataFrame([{
    "test": "One-sided binomial test",
    "null": "Hit rate <= 50%",
    "observations": n,
    "correct_predictions": successes,
    "hit_rate": hit_rate,
    "p_value": hit_pvalue
}])

display(hit_significance)

hit_significance.to_csv(
    TABLES / "sp500_final_test_hit_rate_significance.csv",
    index=False
)


In [ ]:
# Paired daily return test:
# frozen strategy net return versus Buy & Hold daily return.

strategy_returns = final_frame["net_return"].to_numpy()
benchmark_returns = benchmark_returns.to_numpy()

return_difference = (
    strategy_returns -
    benchmark_returns
)

if SCIPY_AVAILABLE:
    paired_test = ttest_rel(
        strategy_returns,
        benchmark_returns
    )
    paired_t_stat = paired_test.statistic
    paired_pvalue = paired_test.pvalue
else:
    d = return_difference
    paired_t_stat = (
        d.mean() /
        (d.std(ddof=1) / np.sqrt(len(d)))
        if d.std(ddof=1) > 0
        else np.nan
    )
    paired_pvalue = np.nan

return_test = pd.DataFrame([{
    "test": "Paired daily-return test",
    "mean_daily_strategy_minus_benchmark": return_difference.mean(),
    "t_statistic": paired_t_stat,
    "p_value": paired_pvalue,
    "strategy_better_mean_return": (
        return_difference.mean() > 0
    )
}])

display(return_test)

return_test.to_csv(
    TABLES / "sp500_final_test_strategy_vs_benchmark_test.csv",
    index=False
)


In [ ]:
# Bootstrap confidence interval for the frozen strategy Sharpe.

rng = np.random.default_rng(RANDOM_SEED)
daily = strategy_returns.copy()
bootstrap_sharpes = []

for _ in range(BOOTSTRAP_SAMPLES):
    sample = rng.choice(
        daily,
        size=len(daily),
        replace=True
    )

    std = sample.std(ddof=1)

    if std > 0:
        bootstrap_sharpes.append(
            sample.mean() /
            std *
            np.sqrt(TRADING_DAYS)
        )

bootstrap_sharpes = np.asarray(
    bootstrap_sharpes
)

sharpe_low, sharpe_high = np.percentile(
    bootstrap_sharpes,
    [2.5, 97.5]
)

sharpe_bootstrap = pd.DataFrame([{
    "observed_sharpe": final_strategy_metrics["Sharpe"],
    "bootstrap_mean_sharpe": bootstrap_sharpes.mean(),
    "CI_2_5_percent": sharpe_low,
    "CI_97_5_percent": sharpe_high,
    "bootstrap_samples": len(bootstrap_sharpes)
}])

display(sharpe_bootstrap)

sharpe_bootstrap.to_csv(
    TABLES / "sp500_final_test_sharpe_bootstrap.csv",
    index=False
)


In [ ]:
# Bootstrap confidence interval for the difference in
# mean daily returns between strategy and benchmark.

difference = return_difference.copy()
bootstrap_differences = []

for _ in range(BOOTSTRAP_SAMPLES):
    sample = rng.choice(
        difference,
        size=len(difference),
        replace=True
    )
    bootstrap_differences.append(
        sample.mean()
    )

bootstrap_differences = np.asarray(
    bootstrap_differences
)

diff_low, diff_high = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)

difference_bootstrap = pd.DataFrame([{
    "observed_mean_difference": difference.mean(),
    "bootstrap_mean_difference": bootstrap_differences.mean(),
    "CI_2_5_percent": diff_low,
    "CI_97_5_percent": diff_high,
    "bootstrap_samples": len(bootstrap_differences)
}])

display(difference_bootstrap)

difference_bootstrap.to_csv(
    TABLES / "sp500_final_test_return_difference_bootstrap.csv",
    index=False
)


In [ ]:
# Final untouched equity curve.

fig = plt.figure(figsize=(15, 7))

plt.plot(
    final_frame["Date"],
    final_frame["equity"],
    label="Frozen Strategy",
    linewidth=2.2
)

plt.plot(
    final_frame["Date"],
    benchmark_equity,
    label="Buy & Hold",
    linewidth=2.2
)

plt.title(
    f"Final Untouched Test — {candidate_model}"
)

plt.xlabel("Date")
plt.ylabel("Equity")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURES / "sp500_final_untouched_equity_curve.png"
fig.savefig(
    path,
    dpi=150,
    bbox_inches="tight"
)
plt.show()

print("Saved:", path)


In [ ]:
# Final untouched drawdown comparison.

fig = plt.figure(figsize=(15, 6))

plt.plot(
    final_frame["Date"],
    final_frame["drawdown"],
    label="Frozen Strategy"
)

plt.plot(
    final_frame["Date"],
    benchmark_dd,
    label="Buy & Hold"
)

plt.title("Final Untouched Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURES / "sp500_final_untouched_drawdown.png"
fig.savefig(
    path,
    dpi=150,
    bbox_inches="tight"
)
plt.show()

print("Saved:", path)


In [ ]:
# Statistical decision framework.
# This is a research conclusion, not a deployment guarantee.

alpha = 0.05

hit_pass = (
    hit_pvalue < alpha and
    hit_rate > 0.50
)

sharpe_ci_above_zero = (
    sharpe_low > 0
)

return_difference_positive = (
    diff_low > 0
)

strategy_sharpe = final_strategy_metrics["Sharpe"]
benchmark_sharpe = benchmark_summary["Sharpe"]

risk_adjusted_better = (
    strategy_sharpe > benchmark_sharpe
)

if (
    hit_pass and
    sharpe_ci_above_zero and
    return_difference_positive and
    risk_adjusted_better
):
    research_verdict = (
        "STRONG POSITIVE EVIDENCE"
    )
elif (
    hit_pass or
    sharpe_ci_above_zero or
    return_difference_positive
):
    research_verdict = (
        "PARTIAL / INCONCLUSIVE EVIDENCE"
    )
else:
    research_verdict = (
        "INSUFFICIENT EVIDENCE OF ROBUST EDGE"
    )

decision_table = pd.DataFrame([{
    "criterion": "Hit rate significantly above 50%",
    "pass": hit_pass,
    "value": hit_rate,
    "p_value": hit_pvalue
}, {
    "criterion": "Bootstrap Sharpe CI entirely above 0",
    "pass": sharpe_ci_above_zero,
    "value": final_strategy_metrics["Sharpe"],
    "p_value": np.nan
}, {
    "criterion": "Bootstrap mean-return difference entirely above 0",
    "pass": return_difference_positive,
    "value": return_difference.mean(),
    "p_value": paired_pvalue
}, {
    "criterion": "Strategy Sharpe > Buy & Hold Sharpe",
    "pass": risk_adjusted_better,
    "value": strategy_sharpe - benchmark_sharpe,
    "p_value": np.nan
}])

display(decision_table)

print("Research verdict:", research_verdict)


In [ ]:
final_test_report = {
    "status": "FINAL_UNTOUCHED_EVALUATION_COMPLETE",
    "candidate_model": candidate_model,
    "final_test_fraction": FINAL_TEST_FRACTION,
    "training_end": train_df["Date"].max().strftime("%Y-%m-%d"),
    "test_start": test_df["Date"].min().strftime("%Y-%m-%d"),
    "test_end": test_df["Date"].max().strftime("%Y-%m-%d"),
    "frozen_configuration": frozen_config,
    "classification_metrics": classification_metrics,
    "strategy_metrics": final_strategy_metrics,
    "benchmark_metrics": benchmark_summary,
    "hit_rate_significance": hit_significance.to_dict(orient="records"),
    "paired_return_test": return_test.to_dict(orient="records"),
    "sharpe_bootstrap": sharpe_bootstrap.to_dict(orient="records"),
    "return_difference_bootstrap": difference_bootstrap.to_dict(orient="records"),
    "decision_table": decision_table.to_dict(orient="records"),
    "research_verdict": research_verdict,
    "methodological_note": (
        "The final test period was held out chronologically and was not used "
        "for model selection or parameter tuning. Statistical tests are "
        "research diagnostics and do not establish future profitability."
    )
}

report_path = (
    REPORTS /
    "sp500_final_untouched_test_report.json"
)

report_path.write_text(
    json.dumps(
        final_test_report,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

display(
    pd.DataFrame([{
        "candidate_model": candidate_model,
        "research_verdict": research_verdict,
        "test_start": test_df["Date"].min().date(),
        "test_end": test_df["Date"].max().date(),
        "test_rows": len(test_df),
        "test_roc_auc": classification_metrics["roc_auc"],
        "test_f1": classification_metrics["f1"],
        "test_CAGR": final_strategy_metrics["CAGR"],
        "test_Sharpe": final_strategy_metrics["Sharpe"],
        "test_max_drawdown": final_strategy_metrics["max_drawdown"]
    }])
)

print("Saved:", report_path)


In [ ]:
# Final integrity checks.

assert train_df["Date"].max() < test_df["Date"].min()
assert test_df["Date"].is_unique
assert final_predictions["Date"].is_unique
assert len(final_predictions) == len(test_df)
assert np.isfinite(test_probability).all()
assert np.isfinite(final_frame["net_return"]).all()

master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

expected_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume"
]

assert list(master_check.columns) == expected_columns

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("FINAL UNTOUCHED TEST INTEGRITY: PASS")
print("Raw master rows:", len(master_check))
print("Final test rows:", len(test_df))
print("No future-date overlap between training and final test.")


# Notebook 13 Complete

### Final stage completed

- Chronological final 15% untouched holdout
- Frozen candidate model retrained only on pre-test data
- Frozen probability threshold
- Frozen transaction-cost assumption
- Frozen slippage assumption
- Final classification metrics
- Final trading metrics
- Buy-and-hold benchmark
- Hit-rate significance test
- Paired strategy-vs-benchmark return test
- Bootstrap Sharpe confidence interval
- Bootstrap return-difference interval
- Final equity and drawdown charts
- Research decision table
- Final research verdict
- JSON final evaluation report
- Final raw-data integrity checks

**Next:** Notebook 14 — Research Summary / Production Model Packaging.

Run Notebook 13 completely and verify the final untouched results before continuing.
